# Notebook 04: Two-Tower Retrieval

## Why This Matters

Modern recommendation systems must retrieve relevant items from catalogs of hundreds of millions. You cannot score every item at query time — that would take seconds. The two-tower architecture solves this by separating the user encoder from the item encoder: pre-compute all item embeddings offline, build an Approximate Nearest Neighbor (ANN) index, and serve recommendations in milliseconds. YouTube's 2019 paper "Sampling-Bias-Corrected Neural Modeling for Large Corpus Item Recommendations" describes exactly this approach at billion-item scale. Google, Spotify, Pinterest, and Twitter all use variants of the two-tower model for their retrieval (candidate generation) stage. This notebook builds the full pipeline: two-tower model in PyTorch + FAISS retrieval index.

## Background

### The Two-Stage Recommendation Pipeline

Most large-scale recommendation systems use two stages:

1. **Retrieval (Candidate Generation)**: From a catalog of millions of items, retrieve the top-$K$ candidates (typically $K \in [100, 1000]$) relevant to the user. Optimized for recall — we don't want to miss good items. This is where the two-tower model lives.

2. **Ranking (Re-ranking)**: Score the $K$ candidates with a more expensive model (features, context, business rules). Optimized for precision. Covered in Notebook 05 (Learning to Rank).

This separation is fundamental: retrieval must be fast (milliseconds, massive fan-out), ranking can be slower (more features, larger model per candidate).

### Two-Tower Architecture

The model has two independent neural networks ("towers"):
- **User tower**: takes user features (user ID, history stats, demographics) → outputs $d$-dimensional embedding $\mathbf{u}$
- **Item tower**: takes item features (item ID, metadata, content) → outputs $d$-dimensional embedding $\mathbf{v}$

Relevance score: $\text{score}(u, i) = \mathbf{u} \cdot \mathbf{v}$ (dot product or cosine similarity)

The towers are independent — during serving, you compute $\mathbf{u}$ once per query and retrieve top-K from a pre-built index of all $\mathbf{v}$ vectors.

### In-Batch Negatives

Training requires positive (user, item) pairs and negative pairs. The trick: treat all *other* items in the same training batch as negatives. If batch size is 1024, each positive pair gets 1023 negatives "for free." This is called **in-batch negative sampling** or **sampled softmax**.

**Popularity bias problem**: Popular items appear more frequently in batches, so they're also more frequently sampled as negatives. The model learns to penalize popular items too much → popular items get lower scores than they deserve. YouTube's correction: subtract $\log(\text{sampling probability})$ from the logits.

### Hard Negatives

In-batch negatives are easy (random items). To improve top-K precision, add **hard negatives** — items that score high for this user but aren't positive. Hard negatives are selected by running an ANN search and taking high-scoring non-positive items. Training alternates between in-batch and hard negatives.

### FAISS: Approximate Nearest Neighbor

FAISS (Facebook AI Similarity Search) builds an index over the pre-computed item embeddings and enables sub-linear nearest neighbor search. Key index types:
- `IndexFlatL2` / `IndexFlatIP`: exact search — $O(n)$, used for small catalogs
- `IndexIVFFlat`: inverted file index — clusters items, only searches top-$n_{probe}$ clusters — $O(\sqrt{n})$
- `IndexHNSWFlat`: hierarchical navigable small world graphs — fast, high recall, no training needed

In [ ]:
%pip install -q torch numpy pandas matplotlib scikit-learn faiss-cpu tqdm

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import faiss
import os, requests, zipfile, io, warnings, time

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 100K ──────────────────────────────────────────────────────
DATA_DIR = "/tmp/ml-100k"
if not os.path.exists(DATA_DIR):
    r = requests.get("https://files.grouplens.org/datasets/movielens/ml-100k.zip", timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")

ratings = pd.read_csv(f"{DATA_DIR}/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])
movies = pd.read_csv(f"{DATA_DIR}/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[0,1], names=["item_id", "title"])

ratings["user_idx"] = ratings.user_id - 1
ratings["item_idx"] = ratings.item_id - 1
n_users = ratings.user_idx.max() + 1
n_items = ratings.item_idx.max() + 1

# Temporal split
ratings = ratings.sort_values(["user_id", "timestamp"])
train_list, test_list = [], []
for uid, group in ratings.groupby("user_id"):
    n = len(group)
    cutoff = max(1, int(n * 0.8))
    train_list.append(group.iloc[:cutoff])
    test_list.append(group.iloc[cutoff:])
train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

user_pos_train = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()
user_pos_test  = test_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

print(f"Users: {n_users}, Items: {n_items}")
print(f"Train: {len(train_df):,}, Test: {len(test_df):,}")
print(f"Device: {device}")
print("Ready.")

## 1. User Features: History Aggregation

The user tower needs features that summarize the user's history. Common choices:
- **Average item embedding** of rated items (bag-of-items representation)
- **Recency-weighted average**: more recent items matter more
- **Count-based features**: how many items rated, rating distribution

For this notebook we'll use two user features:
1. The user ID (learned embedding — captures user-specific preferences)
2. The average of their historical item embeddings (behavior signal — generalizes to new items)

This is sometimes called a **DNN user tower with history pooling** and is the approach used in YouTube DNN (Covington et al., 2016).

In [ ]:
class TwoTowerDataset(Dataset):
    """
    Returns (user_id, pos_item_id, neg_item_ids) for in-batch negative training.
    We return individual samples; the DataLoader collects the batch.
    """
    def __init__(self, train_df, n_items, user_pos_sets):
        self.users = train_df.user_idx.values
        self.items = train_df.item_idx.values
        self.n_items = n_items
        self.user_pos = user_pos_sets

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.users[idx], dtype=torch.long),
            torch.tensor(self.items[idx], dtype=torch.long),
        )


def build_user_history_features(train_df, n_users, n_items):
    """Build a user → list_of_item_ids mapping for history features."""
    user_history = {}
    for uid, group in train_df.groupby("user_idx"):
        # Most recent 50 items, sorted by recency
        items = group.sort_values("timestamp")["item_idx"].tail(50).values
        user_history[uid] = items
    return user_history


user_history = build_user_history_features(train_df, n_users, n_items)

# Item popularity weights for negative sampling
item_counts = np.bincount(train_df.item_idx.values, minlength=n_items).astype(np.float32)
item_pop = item_counts ** 0.75
item_pop /= item_pop.sum()

ds = TwoTowerDataset(train_df, n_items, user_pos_train)
loader = DataLoader(ds, batch_size=512, shuffle=True, num_workers=0, drop_last=True)
print(f"Dataset: {len(ds):,} samples | {len(loader)} batches/epoch")

## 2. Two-Tower Model Architecture

The user tower combines:
1. A learnable user ID embedding
2. The mean of learnable item embeddings from the user's history

The item tower is a simple item ID embedding (could be augmented with content features in production).

Both towers project to the same $d$-dimensional space. The score is a dot product. We train with in-batch negatives: the batch forms a $B \times B$ score matrix, and we want the diagonal (matching user-item pairs) to be highest.

In [ ]:
class UserTower(nn.Module):
    """
    User encoder: user ID embedding + history average pooling.
    Output: L2-normalized embedding of dimension d.
    """
    def __init__(self, n_users, n_items, d=64, history_len=50):
        super().__init__()
        self.user_emb   = nn.Embedding(n_users, d)
        self.item_emb   = nn.Embedding(n_items, d)  # shared with item tower
        self.projection = nn.Sequential(
            nn.Linear(2 * d, d),
            nn.ReLU(),
            nn.Linear(d, d),
        )
        nn.init.normal_(self.user_emb.weight, 0, 0.01)
        nn.init.normal_(self.item_emb.weight, 0, 0.01)

    def forward(self, user_ids, history_items=None):
        """user_ids: (B,) | history_items: (B, L) padded with -1"""
        u = self.user_emb(user_ids)  # (B, d)
        if history_items is not None:
            mask = (history_items >= 0).float().unsqueeze(-1)  # (B, L, 1)
            hist_embs = self.item_emb(history_items.clamp(min=0))  # (B, L, d)
            hist_avg = (hist_embs * mask).sum(1) / mask.sum(1).clamp(min=1)  # (B, d)
            combined = torch.cat([u, hist_avg], dim=1)  # (B, 2d)
            out = self.projection(combined)  # (B, d)
        else:
            out = u
        return F.normalize(out, dim=1)


class ItemTower(nn.Module):
    """Item encoder: item ID embedding → projected + normalized."""
    def __init__(self, n_items, d=64, shared_emb=None):
        super().__init__()
        if shared_emb is not None:
            self.item_emb = shared_emb
        else:
            self.item_emb = nn.Embedding(n_items, d)
            nn.init.normal_(self.item_emb.weight, 0, 0.01)
        self.projection = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, d),
        )

    def forward(self, item_ids):
        e = self.item_emb(item_ids)   # (B, d)
        return F.normalize(self.projection(e), dim=1)


class TwoTowerModel(nn.Module):
    def __init__(self, n_users, n_items, d=64):
        super().__init__()
        self.user_tower = UserTower(n_users, n_items, d)
        self.item_tower = ItemTower(n_items, d, shared_emb=self.user_tower.item_emb)
        self.temperature = nn.Parameter(torch.tensor(0.07))

    def forward(self, user_ids, item_ids, history=None):
        u_emb = self.user_tower(user_ids, history)   # (B, d)
        i_emb = self.item_tower(item_ids)            # (B, d)
        return u_emb, i_emb

    def in_batch_loss(self, u_emb, i_emb):
        """In-batch softmax loss: B×B score matrix, diagonal is the target."""
        temp = torch.clamp(self.temperature, 0.01, 1.0)
        logits = (u_emb @ i_emb.T) / temp   # (B, B)
        labels = torch.arange(len(u_emb), device=u_emb.device)
        loss = F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)
        return loss / 2


model = TwoTowerModel(n_users, n_items, d=64).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Two-Tower Model | Parameters: {n_params:,}")
print(f"Temperature: {model.temperature.item():.4f} (learnable)")

## 3. Training with In-Batch Negatives

We prepare user histories as padded tensors and train end-to-end. The key hyperparameter is **temperature**: lower temperature makes the distribution sharper (more confident), higher temperature makes it softer. Starting around 0.07 (InfoNCE/SimCLR value) works well in practice.

In [ ]:
def collate_with_history(batch, user_history, n_items, max_hist=20):
    """Collate function that adds history features to each batch."""
    users = torch.tensor([b[0] for b in batch], dtype=torch.long)
    items = torch.tensor([b[1] for b in batch], dtype=torch.long)

    # Pad histories
    B = len(batch)
    hist_tensor = torch.full((B, max_hist), -1, dtype=torch.long)
    for i, (u, _) in enumerate(batch):
        u_hist = user_history.get(u.item(), np.array([]))
        hist = u_hist[-max_hist:] if len(u_hist) > 0 else np.array([])
        if len(hist) > 0:
            hist_tensor[i, :len(hist)] = torch.tensor(hist, dtype=torch.long)

    return users, items, hist_tensor


from functools import partial
collate_fn = partial(collate_with_history, user_history=user_history, n_items=n_items)
loader_hist = DataLoader(ds, batch_size=512, shuffle=True,
                         num_workers=0, drop_last=True, collate_fn=collate_fn)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
history_train = []

print("Training Two-Tower model (20 epochs, in-batch negatives)...")
t0 = time.time()
for epoch in range(20):
    model.train()
    total_loss = 0
    for users, items, hist in loader_hist:
        users, items, hist = users.to(device), items.to(device), hist.to(device)
        optimizer.zero_grad()
        u_emb, i_emb = model(users, items, hist)
        loss = model.in_batch_loss(u_emb, i_emb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(loader_hist)
    history_train.append(avg_loss)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/20 | Loss: {avg_loss:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.2e} | Temp: {model.temperature.item():.4f}")

print(f"\nTraining time: {time.time()-t0:.1f}s")

## 4. Building the FAISS Index

After training, we pre-compute embeddings for all items using the item tower, then build a FAISS index. At serving time, we encode the query user and perform an ANN search in the index.

**Index choice tradeoffs**:
- `IndexFlatIP` (exact inner product): $O(n)$ — correct but slow for large catalogs
- `IndexIVFFlat` (inverted file): $O(n/n_{lists})$ — fast, good recall with proper `nprobe`
- `IndexHNSWFlat` (graph): sub-linear, no training needed, high recall, higher memory

For 1682 items, exact search is fine. In production with 10M+ items, you'd use IVF or HNSW.

In [ ]:
# Pre-compute all item embeddings
model.eval()
all_item_ids = torch.arange(n_items, dtype=torch.long).to(device)

with torch.no_grad():
    # Process in batches to avoid memory issues
    batch_size = 256
    all_item_embs = []
    for i in range(0, n_items, batch_size):
        batch_ids = all_item_ids[i:i+batch_size]
        embs = model.item_tower(batch_ids).cpu().numpy()
        all_item_embs.append(embs)
    item_embeddings = np.vstack(all_item_embs).astype(np.float32)

print(f"Item embeddings: {item_embeddings.shape}")
print(f"Embedding norm (should be ~1.0 due to L2 norm): {np.linalg.norm(item_embeddings[0]):.4f}")

# ── Build FAISS index ────────────────────────────────────────────────────────
d = item_embeddings.shape[1]

# Exact inner product search (embeddings are L2-normalized → IP = cosine similarity)
index_flat = faiss.IndexFlatIP(d)
index_flat.add(item_embeddings)
print(f"\nFAISS IndexFlatIP built: {index_flat.ntotal} vectors, dimension {d}")

# IVF index for scalable search (demonstrate on larger use case)
n_lists = 32  # sqrt(n_items) heuristic
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, n_lists, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(item_embeddings)
index_ivf.add(item_embeddings)
index_ivf.nprobe = 8  # search 8 of 32 clusters
print(f"FAISS IndexIVFFlat built: {index_ivf.ntotal} vectors, {n_lists} lists, nprobe={index_ivf.nprobe}")

## 5. End-to-End Retrieval

With the FAISS index built, we can serve recommendations in microseconds per query: encode the user, search the index, filter out seen items, return top-K.

In [ ]:
def retrieve_for_user(user_id, model, index, user_history, user_pos_train,
                      K=50, device="cpu", max_hist=20):
    """Full retrieval pipeline: encode user → ANN search → filter seen items."""
    model.eval()
    u = torch.tensor([user_id], dtype=torch.long).to(device)

    hist = user_history.get(user_id, np.array([]))
    hist_pad = np.full(max_hist, -1)
    if len(hist) > 0:
        hist_pad[:min(len(hist), max_hist)] = hist[-max_hist:]
    hist_t = torch.tensor(hist_pad, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        u_emb = model.user_tower(u, hist_t).cpu().numpy().astype(np.float32)

    # FAISS search: get 2*K candidates to allow for filtering
    D, I = index.search(u_emb, K * 2)
    candidates = I[0]
    scores = D[0]

    # Filter seen items
    seen = user_pos_train.get(user_id, set())
    filtered = [(int(i), float(s)) for i, s in zip(candidates, scores) if i not in seen][:K]

    return filtered


# Demo retrieval for a few users
for uid in [0, 5, 42]:
    t0 = time.time()
    recs = retrieve_for_user(uid, model, index_flat, user_history, user_pos_train, K=10)
    elapsed = (time.time() - t0) * 1000

    rec_items = [r[0] for r in recs[:5]]
    rec_titles = movies[movies.item_id.isin([i+1 for i in rec_items])]["title"].tolist()
    print(f"\nUser {uid}: {elapsed:.2f}ms retrieval time")
    print(f"  Top-5: {rec_titles}")

## 6. Retrieval Quality: Recall@K Evaluation

We measure Recall@K — what fraction of the user's test-set interactions are retrieved in the top-K candidates. This is the metric that matters for retrieval: the goal is high recall so the ranker has good candidates to work with.

In [ ]:
def eval_retrieval_recall(model, index, user_history, user_pos_train, user_pos_test,
                          n_items, K=50, n_eval=300, device="cpu"):
    """Recall@K for retrieval model."""
    recalls = []
    eval_users = [u for u in user_pos_test if len(user_pos_test[u]) > 0][:n_eval]

    for uid in eval_users:
        test_pos = user_pos_test[uid]
        recs = retrieve_for_user(uid, model, index, user_history, user_pos_train,
                                 K=K, device=device)
        rec_set = {r[0] for r in recs}
        hits = len(rec_set & test_pos)
        recalls.append(hits / len(test_pos))

    return np.mean(recalls)


# Compare flat vs IVF
print("Evaluating retrieval recall...")
r_flat_50  = eval_retrieval_recall(model, index_flat, user_history, user_pos_train, user_pos_test, n_items, K=50, device=str(device))
r_flat_100 = eval_retrieval_recall(model, index_flat, user_history, user_pos_train, user_pos_test, n_items, K=100, device=str(device))
r_ivf_50   = eval_retrieval_recall(model, index_ivf,  user_history, user_pos_train, user_pos_test, n_items, K=50, device=str(device))

print(f"\nRetrieval Recall:")
print(f"  IndexFlat  K=50:  {r_flat_50:.4f}")
print(f"  IndexFlat  K=100: {r_flat_100:.4f}")
print(f"  IndexIVF   K=50:  {r_ivf_50:.4f} (approximate, nprobe={index_ivf.nprobe})")
print(f"\nNote: IVF recall is slightly lower — tuning nprobe trades recall for speed.")

# Benchmark search speed
import time
u_emb_test = np.random.randn(1, 64).astype(np.float32)
u_emb_test /= np.linalg.norm(u_emb_test)

n_queries = 1000
t0 = time.time()
for _ in range(n_queries):
    index_flat.search(u_emb_test, 100)
flat_ms = (time.time() - t0) / n_queries * 1000

t0 = time.time()
for _ in range(n_queries):
    index_ivf.search(u_emb_test, 100)
ivf_ms = (time.time() - t0) / n_queries * 1000

print(f"\nSearch latency (1000 queries avg):")
print(f"  IndexFlatIP: {flat_ms:.3f}ms per query")
print(f"  IndexIVFFlat: {ivf_ms:.3f}ms per query  ({flat_ms/ivf_ms:.1f}x speedup)")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Two-stage pipeline | Retrieval (recall) → Ranking (precision) — enables scaling to 100M+ items |
| Two-tower decoupling | User and item encoders are independent → item embeddings pre-computable |
| In-batch negatives | Other items in same batch serve as negatives — cheap but popularity-biased |
| Temperature | Controls sharpness of softmax; $\tau \approx 0.07$ works well (InfoNCE) |
| Hard negatives | High-scoring non-positive items — harder training, better precision |
| FAISS | Vector similarity search library; IndexFlatIP (exact) → IndexIVFFlat/HNSW (approx) |
| Freshness | New items need re-indexed embeddings — typical cadence: hourly or daily |
| Recall@K | Primary metric for retrieval — higher K = higher recall, more ranker work |

### Common Pitfalls
- Not correcting for popularity bias in in-batch negatives — popular items get under-scored
- Evaluating retrieval with Precision@K instead of Recall@K — wrong metric for candidate generation
- Using exact search (IndexFlatIP) in production with millions of items — $O(n)$ latency
- Forgetting to L2-normalize embeddings when using inner product as cosine similarity
- Not rebuilding the item index after model updates — stale embeddings serve stale recommendations